In [50]:
import pandas as pd
import torch
import numpy as np
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from transformers import AutoTokenizer

In [51]:
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
1
NVIDIA GeForce RTX 3050 Laptop GPU


In [52]:
# Read the file
df = pd.read_csv(r'/GenAI Project-(IITM)/train.csv')
test_df = pd.read_csv(r'/GenAI Project-(IITM)/test.csv')
df

,id,text,anger,fear,joy,sadness,surprise,emotions
0,0,the dentist that did the work apparently did a...,1,0,0,1,0,['anger' 'sadness']
1,1,i'm gonna absolutely ~~suck~~ be terrible duri...,0,1,0,1,0,['fear' 'sadness']
2,2,"bridge: so leave me drowning calling houston, ...",0,1,0,1,0,['fear' 'sadness']
3,3,after that mess i went to see my now ex-girlfr...,1,1,0,1,0,['anger' 'fear' 'sadness']
4,4,"as he stumbled i ran off, afraid it might some...",0,1,0,0,0,['fear']
...,...,...,...,...,...,...,...,...
6822,6822,there is not a cloud in the sky and the sun is...,0,0,1,0,0,['joy']
6823,6823,&gt; the grave stomper,0,0,0,0,1,['surprise']
6824,6824,my ear was still freaking stuck.,1,1,0,0,0,['anger' 'fear']
6825,6825,i felt like there was an electric current flow...,0,1,0,1,0,['fear' 'sadness']


In [53]:
print("Checking the info :", df.info())
print("Checking for null values :", df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6827 entries, 0 to 6826
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        6827 non-null   int64 
 1   text      6827 non-null   object
 2   anger     6827 non-null   int64 
 3   fear      6827 non-null   int64 
 4   joy       6827 non-null   int64 
 5   sadness   6827 non-null   int64 
 6   surprise  6827 non-null   int64 
 7   emotions  6827 non-null   object
dtypes: int64(6), object(2)
memory usage: 426.8+ KB
Checking the info : None
Checking for null values : id          0
text        0
anger       0
fear        0
joy         0
sadness     0
surprise    0
emotions    0
dtype: int64


### LABEL CHARACTERISTICS


In [54]:
labels = ['anger','fear','joy','sadness','surprise']
df[labels].mean().sort_values()

anger       0.118354
joy         0.243152
surprise    0.292808
sadness     0.318002
fear        0.565402
dtype: float64

### MULTI-LABEL COMPLEXITY
#### 1.How many samples have 0, 1, 2, 3+ emotions?
#### 2.Are some emotions almost always co-occurring?

In [55]:
df['num_labels'] = df[labels].sum(axis=1)
df['num_labels'].value_counts(normalize=True)

num_labels
1    0.401787
2    0.378937
3    0.103413
0    0.099019
4    0.016405
5    0.000439
Name: proportion, dtype: float64

### TEXT LENGTH DISTRIBUTION


In [56]:
df['text_len'] = df['text'].str.split().apply(len)
df['text_len'].describe()

count    6827.000000
mean       15.684781
std        11.365744
min         1.000000
25%         8.000000
50%        13.000000
75%        20.000000
max        89.000000
Name: text_len, dtype: float64

### Project Overview

This project builds a multi-label emotion classification system for short English text, targeting five emotions (anger, fear, joy, sadness, surprise) and optimized for Macro F1-score under extreme class imbalance. The workflow begins with targeted EDA to analyze label co-occurrence, imbalance, and text length, followed by minimal transformer-friendly preprocessing and stratified train–validation splitting. A transformer-based baseline model (RoBERTa) is trained to capture high-frequency emotion patterns, with experiments tracked using MLflow. To improve performance on rare emotions, the pipeline incorporates class-conditional LLM-based data augmentation, followed by per-label threshold optimization to directly maximize Macro F1. The final model is registered via MLflow and deployed through a Streamlit web application that provides real-time emotion predictions with confidence scores, demonstrating an end-to-end, production-oriented NLP system.

### DATA PREPROCESS

#### Defining labels and device

In [57]:
LABELS = ["anger", "fear", "joy", "sadness", "surprise"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [58]:
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = text.strip()
    text = " ".join(text.split())
    return text
df['text'] = df['text'].apply(clean_text)

### TRAIN AND VALIDATION SPLIT

In [59]:
# Stratify the labels
df['label_bins'] = pd.qcut(
    df['num_labels'],
    q=4,
    duplicates='drop'
)

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label_bins'],
    random_state=42
)

In [60]:
print('Train data shape:', train_df.shape)
print('Validation data shape:', val_df.shape)

Train data shape: (5461, 11)
Validation data shape: (1366, 11)


In [61]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5461 entries, 1783 to 1196
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   id          5461 non-null   int64   
 1   text        5461 non-null   object  
 2   anger       5461 non-null   int64   
 3   fear        5461 non-null   int64   
 4   joy         5461 non-null   int64   
 5   sadness     5461 non-null   int64   
 6   surprise    5461 non-null   int64   
 7   emotions    5461 non-null   object  
 8   num_labels  5461 non-null   int64   
 9   text_len    5461 non-null   int64   
 10  label_bins  5461 non-null   category
dtypes: category(1), int64(8), object(2)
memory usage: 474.8+ KB


### TOKENIZATION

In [62]:
MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        list(batch),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

### DATASET CLASS

In [63]:
class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_length=128, has_labels=True):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.iloc[idx]["text"]

        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
        }

        if self.has_labels:
            labels = self.df.loc[self.df.index[idx], LABELS].values.astype(float)
            item["labels"] = torch.tensor(labels, dtype=torch.float)

        return item

### DATA LOADERS

In [64]:
train_dataset = EmotionDataset(train_df, tokenizer)
val_dataset = EmotionDataset(val_df, tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

In [65]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    problem_type="multi_label_classification"
)

model.to(DEVICE)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [66]:
# Compute class weights
pos_counts = train_df[LABELS].sum().values
neg_counts = len(train_df) - pos_counts

pos_weights = neg_counts / pos_counts

print("Positive weights:", pos_weights)

Positive weights: [7.33740458 0.75934278 3.15285171 2.15118292 2.4216792 ]


### LOSS & OPTIMIZER

In [67]:
loss_fn = torch.nn.BCEWithLogitsLoss()

optimizer = AdamW(model.parameters(), lr=1e-5)

In [68]:
scaler = torch.amp.GradScaler("cuda")

### TRAINING LOOP

In [69]:
best_f1 = 0
patience = 2
counter = 0

In [70]:
# for epoch in range(5):
#
#     model.train()
#     total_loss = 0
#
#     for batch in train_loader:
#         batch = {k: v.to(DEVICE) for k, v in batch.items()}
#
#         optimizer.zero_grad()
#
#         with torch.amp.autocast("cuda"):
#             outputs = model(
#                 input_ids=batch["input_ids"],
#                 attention_mask=batch["attention_mask"]
#             )
#             loss = loss_fn(outputs.logits, batch["labels"])
#
#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()
#
#         total_loss += loss.item()
#
#     print(f"Epoch {epoch+1} | Train Loss: {total_loss:.4f}")
#
#     model.eval()
#
#     all_preds = []
#     all_labels = []
#
#     with torch.no_grad():
#         for batch in val_loader:
#             batch = {k: v.to(DEVICE) for k, v in batch.items()}
#
#             outputs = model(
#                 input_ids=batch["input_ids"],
#                 attention_mask=batch["attention_mask"]
#             )
#
#             probs = torch.sigmoid(outputs.logits)
#             preds = (probs > 0.5).float()
#
#             all_preds.append(preds.cpu())
#             all_labels.append(batch["labels"].cpu())
#
#     all_preds = torch.cat(all_preds)
#     all_labels = torch.cat(all_labels)
#
#     macro_f1 = f1_score(all_labels, all_preds, average="macro")
#
#     print(f"Epoch {epoch+1} | Val Macro F1: {macro_f1:.4f}")

### SAVE THE BEST MODEL (Hugging face format)

In [71]:
model.save_pretrained("best_roberta_emotion")
tokenizer.save_pretrained("best_roberta_emotion")

('best_roberta_emotion\\tokenizer_config.json',
 'best_roberta_emotion\\special_tokens_map.json',
 'best_roberta_emotion\\vocab.json',
 'best_roberta_emotion\\merges.txt',
 'best_roberta_emotion\\added_tokens.json',
 'best_roberta_emotion\\tokenizer.json')

### DEFINE THE DEVICE

In [72]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### LOADING THE PRETRAINED MODEL

In [73]:
from transformers import RobertaForSequenceClassification, RobertaTokenizer

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=5
)

model.load_state_dict(torch.load("../models/best_roberta_emotion.pt"))
model.to(DEVICE)
model.eval()

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\RAHUL.ASUS-VIVOBOOK-P\AppData\Local\Temp\ipykernel_11544\3352765702.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded 

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

### VALIDATION PROBABILITIES

In [74]:
model.eval()

val_probs = []
val_true = []

with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        ).logits

        probs = torch.sigmoid(logits).cpu().numpy()

        val_probs.append(probs)
        val_true.append(batch["labels"].cpu().numpy())

val_probs = np.vstack(val_probs)
val_true = np.vstack(val_true)

### SETTING THE THRESHOLD

In [75]:
best_thresholds = []

for i in range(len(LABELS)):
    best_f1 = 0
    best_t = 0.5

    for t in np.arange(0.1, 0.9, 0.05):
        preds = (val_probs[:, i] > t).astype(int)
        f1 = f1_score(val_true[:, i], preds)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    best_thresholds.append(best_t)

print("Best thresholds:", best_thresholds)

Best thresholds: [np.float64(0.45000000000000007), np.float64(0.6500000000000001), np.float64(0.6000000000000002), np.float64(0.6000000000000002), np.float64(0.30000000000000004)]


### VALIDATION (Macro - F1)

In [76]:
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        ).logits

        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs > 0.5).astype(int)

        all_preds.append(preds)
        all_labels.append(batch["labels"].cpu().numpy())

y_pred = np.vstack(all_preds)
y_true = np.vstack(all_labels)

print("Validation Macro F1:", f1_score(y_true, y_pred, average="macro"))

Validation Macro F1: 0.8231060586828567


### PER LABEL MACRO F1

In [77]:
print(classification_report(y_true, y_pred, target_names=LABELS))

              precision    recall  f1-score   support

       anger       0.75      0.75      0.75       153
        fear       0.86      0.89      0.88       756
         joy       0.84      0.86      0.85       345
     sadness       0.79      0.88      0.83       438
    surprise       0.84      0.78      0.81       403

   micro avg       0.83      0.85      0.84      2095
   macro avg       0.82      0.83      0.82      2095
weighted avg       0.83      0.85      0.84      2095
 samples avg       0.77      0.77      0.75      2095



D:\Practice Projects\NLP - Projects\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
D:\Practice Projects\NLP - Projects\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
D:\Practice Projects\NLP - Projects\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len

## Now we do the testing on the test set we have

### CREATE THE TEST DATA SET

In [78]:
test_dataset = EmotionDataset(
    test_df,
    tokenizer,
    max_length=128,
    has_labels=False
)

### TEST DATA LOADER

In [79]:
test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

### EVALUATE ON TEST SET

In [80]:
all_probs = []

model.eval()

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        ).logits

        probs = torch.sigmoid(logits).cpu().numpy()

        all_probs.append(probs)

# Now stack properly
all_probs = np.vstack(all_probs)

print("Final shape:", all_probs.shape)

Final shape: (1707, 5)


In [81]:
y_pred = np.zeros_like(all_probs)

for i, t in enumerate(best_thresholds):
    y_pred[:, i] = (all_probs[:, i] > t).astype(int)

print(y_pred.shape)

(1707, 5)


In [82]:
# all_preds = []
# all_probs = []
#
# with torch.no_grad():
#     for batch in test_loader:
#         batch = {k: v.to(DEVICE) for k, v in batch.items()}
#
#         logits = model(
#             input_ids=batch["input_ids"],
#             attention_mask=batch["attention_mask"]
#         ).logits
#
#         probs = torch.sigmoid(logits).cpu().numpy()
#         preds = (probs > 0.5).astype(int)
#
#         all_probs.append(probs)
#         all_preds.append(preds)
#
# y_pred = np.vstack(all_preds)

### FOR TESTING ON KAGGLE

In [83]:
y_pred

array([[1., 1., 0., 1., 0.],
       [0., 0., 0., 0., 0.],
       [1., 1., 0., 0., 1.],
       ...,
       [0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 1.]], shape=(1707, 5), dtype=float32)

In [84]:
submission = pd.DataFrame(
    y_pred,
    columns=LABELS
)

submission.insert(0, "id", test_df["id"].values)
print(submission)

        id  anger  fear  joy  sadness  surprise
0        0    1.0   1.0  0.0      1.0       0.0
1        1    0.0   0.0  0.0      0.0       0.0
2        2    1.0   1.0  0.0      0.0       1.0
3        3    0.0   1.0  0.0      0.0       0.0
4        4    0.0   1.0  0.0      0.0       1.0
...    ...    ...   ...  ...      ...       ...
1702  1702    0.0   1.0  1.0      0.0       1.0
1703  1703    0.0   0.0  0.0      0.0       0.0
1704  1704    0.0   1.0  0.0      0.0       0.0
1705  1705    0.0   0.0  0.0      0.0       0.0
1706  1706    0.0   1.0  0.0      0.0       1.0

[1707 rows x 6 columns]


In [85]:
# Save it as CSV
submission.to_csv("submission.csv", index=False)